# בניית אינדקס חיפוש חכם ב־Kaggle (שני T4)

המחברת מורידה את שני חלקי `seforim_bundle.tar.zst` מתוך [גרסת מסד הנתונים v2-20260814115718](https://github.com/arieldaniely/SeforimLibrary/releases/tag/v2-20260814115718), מאמתת SHA־256, ומחלצת רק את `seforim.db`. לאחר מכן היא מקודדת את השורות בשני GPU, בונה שמונה חלקי Lucene, ואורזת את האינדקס והמודל ל־`semantic-bundle.tar.zst` (או לחלקים) ול־`semantic-bundle.json`.

לפני ההרצה: בחרו **Accelerator → GPU T4 x2**, הפעילו **Internet**, והוסיפו ב־**Add-ons → Secrets** סוד בשם `HF_TOKEN` עם גישה מאושרת לשני מאגרי המודל. אפשר להריץ `Save Version → Save & Run All` כדי שהעבודה תמשיך גם כשהדפדפן נסגר. החבילה שתיווצר תתאים למסד הנתונים של ה־Release הזה; אם המסד המותקן שונה ממנו, Zayit יסרב לייבא אותה.


In [ ]:
%pip install -q zstandard huggingface_hub safetensors onnxruntime 'transformers<5' requests


In [ ]:
import torch
import shutil
from pathlib import Path

assert torch.cuda.device_count() == 2, 'בחרו GPU T4 x2 בהגדרות המחברת'
print([torch.cuda.get_device_name(i) for i in range(2)])
print('Free scratch space (GiB):', round(shutil.disk_usage('/kaggle/temp').free / 2**30, 1))


In [ ]:
import subprocess

repo = Path('/kaggle/working/Zayit')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/semantic-search-round2',
                    'https://github.com/arieldaniely/Zayit.git', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'submodule', 'update', '--init', 'SeforimLibrary'], check=True)
assert (repo / 'SeforimLibrary/search/src/jvmMain/kotlin/io/github/kdroidfilter/seforimlibrary/search/BuildSemanticIndexFromVectors.kt').is_file()
print('Code:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import sys

subprocess.run([sys.executable, str(repo / 'scripts/kaggle_semantic_bundle.py'),
                '--repo', str(repo), '--work', '/kaggle/temp/zayit-semantic',
                '--output', '/kaggle/working', '--batch', '128'], check=True)


In [ ]:
for path in sorted(Path('/kaggle/working').glob('semantic-bundle*')):
    print(path.name, f'{path.stat().st_size / 2**30:.2f} GiB')
print((Path('/kaggle/working') / 'checksums.sha256').read_text())


## לאחר סיום

הורידו מ־**Output** את `semantic-bundle.json`, את קובץ `semantic-bundle.tar.zst` היחיד או את כל קובצי `semantic-bundle.tar.zst.partXX`, ואת `checksums.sha256`. לייבוא מקומי ב־Zayit בחרו רק את קובץ ה־zst או את כל חלקיו. לפרסום כ־Release צרפו גם את קובץ המניפסט; החבילה לא תופיע להורדה אוטומטית מהאפליקציה כל עוד ה־Release נשאר טיוטה.
